# N2T Python Review: f-strings, File I/O, and OO Design
stough F26

A quick refresher on three Python tools you'll lean on heavily for the upcoming
Assembler and VM Translator and compiler projects: building formatted strings,
reading and writing files, and organizing code with classes. None of the
examples below do the actual project work for you — the goal is just to get
the syntax back in your fingers so it's not a surprise later.

More basic Python reviews:
- [Learn Python](https://www.learnpython.org/): through at least "Learn the Basics"
- [codecademy Python 3](https://www.codecademy.com/learn/learn-python-3)
- [W3 Schools Python](https://www.w3schools.com/python/)
- [Basic Python Elements](https://github.com/joshuastough/pythonstructures/blob/main/ch00/0_basic_python_elements.ipynb): this loser's repo has some dead links but does show lots of examples for each of the six basic elements of any good programming language: variables, expressions, conditional execution, iteration, functional and data abstraction.

This notebook assumes some experience related to the above sources and jumps right in to
string manipulation, file I/O, and object-oriented design relevant to the later projects
of N2T, including P6 assembler, P7-8 VM translator, and P9-10 compilers.


## 1. String Manipulation

Strings are immutable character sequences delineated by quotes, `'` or `"`. Python offers a number of useful for parsing, manipulating and formatting these strings. Here's a brief look at a few

In [1]:
# split returns a list of tokens in the sting, where tokens are separated by the specified delimiter.
s = 'hey now brown cow'
print(s.split())

s = 'Another example, with commas, even a semicolon; and other stuff...'
print(s.split(','))

['hey', 'now', 'brown', 'cow']
['Another example', ' with commas', ' even a semicolon; and other stuff...']


In [2]:
s = '   strangely formatted string    with    lots of whitespace   '
print("'" + s + "'")
print("'" + s.strip() + "'" + " after strip()")
print(s.split())

'   strangely formatted string    with    lots of whitespace   '
'strangely formatted string    with    lots of whitespace' after strip()
['strangely', 'formatted', 'string', 'with', 'lots', 'of', 'whitespace']


In [3]:
s = '   MD=D+1  // stuff'
s.strip()

'MD=D+1  // stuff'

In [4]:
temp = s.strip()
temp.split('=')

['MD', 'D+1  // stuff']

In [5]:
temp.split('=')[1].split()

['D+1', '//', 'stuff']

## 2. f-strings and `.format`

An f-string (`f"..."`) lets you drop Python expressions directly inside a
string using `{ }`. Alternatively, `str.format(...)` does something similar
but with the values passed in after the string instantiation. Both support a
"format spec" after a `:` for things like padding, alignment, fixed width,
and number base (binary, hex, etc.) — very handy any time you need to print
things in neat columns or in a particular numeric representation.

In [6]:
name = "amp"
price = 249.5
qty = 3

# basic f-string
print(f"{qty} x {name} = ${price * qty:.2f}")

# same thing with .format
print("{} x {} = ${}".format(qty, name, price * qty))


3 x amp = $748.50
3 x amp = $748.5


Format specs get more interesting once you add width, fill characters, and
alignment. This is the same machinery you'd use to line up a printed table,
zero-pad a number, or show a value in a different base.

In [7]:
inventory = {"widgets": 7, "gizmos": 42, "bolts": 128}

for item, count in inventory.items():
    # :<10 left-aligns in a 10-char field, :>4 right-aligns in a 4-char field
    print(f"{item:<10}{count:>4}")


widgets      7
gizmos      42
bolts      128


In [8]:
# number formatting: width, zero-fill, and base all live in the format spec
for n in [3, 42, 255]:
    print(f"{n:>3} in binary is {n:08b}, in hex is {n:#06x}")


  3 in binary is 00000011, in hex is 0x0003
 42 in binary is 00101010, in hex is 0x002a
255 in binary is 11111111, in hex is 0x00ff


Your turn: given a list of (name, score) tuples, print a right-aligned report where names are left-justified in a 12-char field and scores are zero-padded to 3 digits. Try it with both an f-string and `.format`.

In [9]:
scores = [("Ada", 98), ("Grace", 7), ("Alan", 100)]

# TODO: print each as   Ada         098


### You can also pre-format strings for later use
You can nest `{{}}` to avoid having to `format` everything at the same time.  

In [10]:
s = '{} is {{}} years old'
print(s.format("Alice"))
print(s.format('Alice').format(42))
print(s.format("Bob").format(99))

Alice is {} years old
Alice is 42 years old
Bob is 99 years old


## 3. File I/O with `with open(...) as f:`

The `with` statement gives you a file handle that automatically closes when
the block ends, even if an error happens inside it. Reading a file line by
line and writing a new file line by line is a pattern you'll use constantly
once you're translating one file format into another.

In [15]:
# create a small sample file to work with
with open("sample_input.txt", "w") as f:
    f.write("// a sample file\n")
    f.write("apple, 3\n")
    f.write("\n")
    f.write("banana, 5\n")
    f.write("// end of file\n")


Reading it back: `for line in f` walks the file one line at a time. It's
common to strip whitespace, skip blank lines and comments, and split the
remaining content into fields before doing anything with it.

In [16]:
with open("sample_input.txt", "r") as f:
    for raw_line in f:
        line = raw_line.strip()
        if line == "" or line.startswith("//"):
            continue
        fruit, count = line.split(",")
        print(f"{fruit.strip():<10}{int(count):>3}")


apple       3
banana      5


Now write the processed results to a *new* file instead of just printing them. Open one file for reading and a second file for writing, and copy over a transformed version of each valid line.

In [17]:
with open("sample_input.txt", "r") as fin, open("sample_output.txt", "w") as fout:
    for raw_line in fin:
        line = raw_line.strip()
        if line == "" or line.startswith("//"):
            continue
        fruit, count = line.split(",")
        fout.write(f"{fruit.strip():<10}{int(count):>3}\n")
        


## 4. A bit of OO design

Once you're generating output files with any structure (comments, headers,
repeated formatting, a running count of what's been written), it's worth
wrapping that behavior in a class instead of scattering `open`/`write` calls
around your script. The object can hold the open file handle as state, and
expose a few well-named methods that do the formatting for you.

Below is a small `FileWriter` example — not tied to any particular project,
just a shape you'll recognize later when you build something similar for
real output files.

In [ ]:
class FileWriter:
    def __init__(self, filename):
        self.filename = filename
        self.f = open(filename, "w")
        self.line_count = 0

    def write_line(self, text):
        self.f.write(f"{text}\n")
        self.line_count += 1

    def write_comment(self, text):
        self.write_line(f"// {text}")

    def write_labeled(self, label, value, width=10):
        self.write_line(f"{label:<{width}}{value}")

    def close(self):
        print(f"wrote {self.line_count} lines to {self.filename}")
        self.f.close()


In [ ]:
fw = FileWriter("demo_output.txt")
fw.write_comment("generated by FileWriter demo")
fw.write_labeled("apple", 3)
fw.write_labeled("banana", 5)
fw.close()

with open("demo_output.txt") as f:
    print(f.read())


Notice `FileWriter` is just bundling the ideas from sections 1 and 2: it
keeps an open file handle as an attribute, uses f-strings internally to
format what gets written, and tracks a bit of its own state (`line_count`)
across multiple method calls. A class like this becomes especially useful
once "writing output" turns into a multi-step process you call many times
from different places in your code.

Your turn: add a method `write_many(self, items)` that takes a list of `(label, value)` pairs and calls `write_labeled` on each one. Then use it to rewrite the `fw.write_labeled(...)` calls above as a single call.

In [ ]:
# TODO: extend FileWriter with write_many, then try it out
